In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [2]:
import pandas as pd
import glob
import os
from datasets import Dataset
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_PATH = "../"
CLEANING_DATASET = BASE_PATH + "data/cleaning/"
MODEL_SAVE_PATH = BASE_PATH + "models/rut5-cleaner/"
TUNED_MODEL_SAVE_PATH = BASE_PATH + "models/rut5-cleaner-tuned/"

csv_files = sorted(glob.glob(CLEANING_DATASET + "*.csv"))
dfs = [pd.read_csv(f) for f in csv_files]
full_df = pd.concat(dfs, ignore_index=True)

full_df = full_df.dropna().reset_index(drop=True)

dataset = Dataset.from_pandas(full_df)
dataset = dataset.train_test_split(test_size=0.1)

print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

/workspace/lecture-transcriber/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train size: 606
Test size: 68


In [4]:
from transformers import T5Tokenizer

model_checkpoint = "cointegrated/rut5-base"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint, legacy=False, token=HF_TOKEN)

max_input_length = 512
max_target_length = 512

def preprocess_function(examples):
    inputs = ["clean: " + doc for doc in examples["raw_text"]]
    model_inputs = tokenizer(
        inputs, 
        max_length=512, 
        truncation=True, 
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"], 
        max_length=512, 
        truncation=True, 
        padding="max_length"
    )

    labels_with_ignore_index = []
    for labels_example in labels["input_ids"]:
        labels_example = [
            (l if l != tokenizer.pad_token_id else -100) 
            for l in labels_example
        ]
        labels_with_ignore_index.append(labels_example)

    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 68/68 [00:00<00:00, 360.26 examples/s]


In [5]:
def check_token_lengths(dataset, tokenizer, max_val=512):
    lengths = [len(tokenizer.encode(x)) for x in dataset['raw_text']]
    oversized = [l for l in lengths if l > max_val]
    
    print(f"Avg length: {sum(lengths)/len(lengths):.1f} tokens")
    print(f"Max length: {max(lengths)} tokens")
    print(f"Percentage of cutted raws: {len(oversized)/len(lengths)*100:.2f}%")

check_token_lengths(full_df, tokenizer)

Avg length: 331.4 tokens
Max length: 557 tokens
Percentage of cutted raws: 7.27%


In [6]:
torch.cuda.empty_cache()

In [16]:
import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_checkpoint = "cointegrated/rut5-base"

config = AutoConfig.from_pretrained(model_checkpoint, token=HF_TOKEN)
config.tie_word_embeddings = False 
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, config=config, token=HF_TOKEN)

model.encoder.embed_tokens.weight = model.shared.weight
model.decoder.embed_tokens.weight = model.shared.weight

# model.gradient_checkpointing_enable()


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="epoch", 
    save_strategy="epoch",

    learning_rate=5e-4,   
    num_train_epochs=15, 
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16, 
    
    weight_decay=0.01,
    save_total_limit=1,
    
    bf16=True,
    optim="adafactor",
    lr_scheduler_type="cosine",
    warmup_steps=80,
    
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

torch.cuda.empty_cache()

trainer.train()

Epoch,Training Loss,Validation Loss
1,16.821100,11.591211
2,12.930200,6.023413
3,6.880400,2.976742
4,3.463700,2.381394
5,2.827900,2.178669
6,2.564900,2.112399
7,2.391700,2.044685
8,2.196000,1.960284
9,2.052100,1.957608
10,1.908300,1.829523


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=135, training_loss=4.428802801061559, metrics={'train_runtime': 328.7039, 'train_samples_per_second': 27.654, 'train_steps_per_second': 0.411, 'total_flos': 5572482045050880.0, 'train_loss': 4.428802801061559, 'epoch': 13.526315789473685})

In [18]:
model.save_pretrained(TUNED_MODEL_SAVE_PATH)
tokenizer.save_pretrained(TUNED_MODEL_SAVE_PATH)

('../models/rut5-cleaner-tuned/tokenizer_config.json',
 '../models/rut5-cleaner-tuned/special_tokens_map.json',
 '../models/rut5-cleaner-tuned/spiece.model',
 '../models/rut5-cleaner-tuned/added_tokens.json',
 '../models/rut5-cleaner-tuned/tokenizer.json')

In [19]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(TUNED_MODEL_SAVE_PATH)
config = AutoConfig.from_pretrained(TUNED_MODEL_SAVE_PATH)
config.tie_word_embeddings = False 
model = AutoModelForSeq2SeqLM.from_pretrained(TUNED_MODEL_SAVE_PATH, config=config)
model.encoder.embed_tokens.weight = model.shared.weight
model.decoder.embed_tokens.weight = model.shared.weight

/workspace/lecture-transcriber/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [20]:
import time

def clean_text(text, model, tokenizer):
    model.eval()
    device = model.device
    input_text = "clean: " + text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=512,
            num_beams=5,  
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True
        )
    
    return  tokenizer.decode(outputs[0], skip_special_tokens=True)

raw_text = dataset['train'][0]['raw_text']
print(raw_text)
print(clean_text(raw_text, model, tokenizer))

Ничего не понятно. А нам бы можно было пропустить Thread 2? Thread 2? Ну, а четверка тогда бы что ждал? Ну, в смысле четверка бы... Ну, в четверке было бы SYNC 1. Да, можно было. Здесь SYNC 1, да, Alload. Можно было пропустить Thread 2, но тут просто пример, что можно вот так вот делать цепочки через Thread, который вообще, как говорят, не трогает никак данные. И вернемся к Mute. Я надеюсь, мы сможем понять, как он работает. Ну, в общем, там все в атомиках и мемориорах. Короче, Mutex можно реализовывать по-разному. Вот, я видела реализацию, ну, кажется, писала реализацию, когда там один атомик, и у него... Короче, структура Mutex, она стоит из нескольких полей, там не только есть, там еще несколько полей есть. Тут по ссылке можете, когда я выложу презентацию, потыкаться. Вот, но основное поле, которое нас волнует, это интовое поле, которое называется Lock. В общем, оно примерно имеет три состояния. 0, 1 и 2. 0 — это значит, что Mutex свободен, 1 значит, он занят, и 2 значит, у нас есть

В четверке было бы SYNC 1. Можно было пропустить Thread 2, но в четверке было бы SYNC 1. Существует три состояния: 0, 1 и 2. 0 означает, что Mutex свободен, 1 — занят, 2 — ожидающие потоки. Для выполнения цепочек через Thread используется интовое поле Lock, которое имеет три состояния: 0, 1 и 2. 0 означает, что Mutex свободен, 1 — занят, 2 — заняты потоки.
